1. Start simple while training a simple architecture
2. Grid Search/Random Search: Use grid search or random search to try different architectures.
3. Cross-validation: Use cross-validation to evaluate the performance of different architectures.
4. Heuristics and Rules of thumb: Some heuristics and smpirical rules can provide starting points such as:
- The number of neurons in the hidden layer should be between the size of the input layer and the size of the output layer
- a common practice is to start with 1-2 hidden layers

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
import pickle

In [4]:
data=pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

In [5]:
## Define a function to create the model and try different parameters(KerasClassifier)

def create_model(neurons=32,layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model

In [6]:
## Create a Keras classifier
from scikeras.wrappers import KerasClassifier

model = KerasClassifier(
    model=create_model,
    verbose=0
)

In [7]:
# Define the grid search parameters
param_grid = {
    'model__neurons': [16, 32, 64, 128],
    'model__layers': [1, 2],
    'epochs': [50, 100]
}

In [9]:
# Perform grid search
import itertools

neurons_list = [16, 32, 64, 128]
layers_list = [1, 2]
epochs_list = [50, 100]

best_accuracy = 0
best_params = None

for neurons, layers, epochs in itertools.product(neurons_list, layers_list, epochs_list):
    print(f"Testing: neurons={neurons}, layers={layers}, epochs={epochs}")
    
    model = create_model(neurons=neurons, layers=layers)
    
    model.fit(X_train, y_train, epochs=epochs, batch_size=32, verbose=0)
    
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    
    print(f"Accuracy: {accuracy:.4f}")
    
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_params = (neurons, layers, epochs)

print("\nBest Accuracy:", best_accuracy)
print("Best Params (neurons, layers, epochs):", best_params)

Testing: neurons=16, layers=1, epochs=50


2026-04-27 16:09:09.780903: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


Accuracy: 0.8120
Testing: neurons=16, layers=1, epochs=100
Accuracy: 0.8140
Testing: neurons=16, layers=2, epochs=50
Accuracy: 0.7790
Testing: neurons=16, layers=2, epochs=100
Accuracy: 0.7030
Testing: neurons=32, layers=1, epochs=50
Accuracy: 0.8145
Testing: neurons=32, layers=1, epochs=100
Accuracy: 0.8020
Testing: neurons=32, layers=2, epochs=50
Accuracy: 0.8010
Testing: neurons=32, layers=2, epochs=100
Accuracy: 0.7365
Testing: neurons=64, layers=1, epochs=50
Accuracy: 0.8165
Testing: neurons=64, layers=1, epochs=100
Accuracy: 0.8190
Testing: neurons=64, layers=2, epochs=50
Accuracy: 0.7500
Testing: neurons=64, layers=2, epochs=100
Accuracy: 0.7260
Testing: neurons=128, layers=1, epochs=50
Accuracy: 0.8110
Testing: neurons=128, layers=1, epochs=100
Accuracy: 0.8105
Testing: neurons=128, layers=2, epochs=50
Accuracy: 0.6905
Testing: neurons=128, layers=2, epochs=100
Accuracy: 0.7755

Best Accuracy: 0.8190000057220459
Best Params (neurons, layers, epochs): (64, 1, 100)
